# 04 — Визуализация результатов сегментации

Качественный анализ масок: сравнение ground truth vs предсказания модели.

In [ ]:
import sys; sys.path.insert(0, '..')
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from omegaconf import OmegaConf

from src.models.model_factory import create_model
from src.data.transforms import build_val_transforms
from src.pipeline.stage1_segment import run_segmentation
from src.utils.visualization import overlay_masks

# Load model
cfg = OmegaConf.merge(
    OmegaConf.load('../configs/base.yaml'),
    OmegaConf.load('../configs/unet.yaml'),
)
CKPT = sorted(Path('../checkpoints').glob('unet-*.ckpt'))[-1]
model = create_model(cfg)
state = torch.load(CKPT, map_location='cpu')
model.load_state_dict(state.get('state_dict', state))
model.eval()
print(f'Loaded checkpoint: {CKPT}')

In [ ]:
transform = build_val_transforms(cfg)
IMG_DIR = Path('../data/processed/images')
MASK_DIR = Path('../data/processed/masks')

test_ids = Path('../data/splits/test.txt').read_text().strip().splitlines()[:6]

fig, axes = plt.subplots(3, len(test_ids), figsize=(18, 9))
for i, sid in enumerate(test_ids):
    img_path = IMG_DIR / f'{sid}.png'
    img  = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    mask_gt = cv2.imread(str(MASK_DIR / f'{sid}.png'), cv2.IMREAD_GRAYSCALE)

    aug = transform(image=img, mask=mask_gt)
    tensor = aug['image'].unsqueeze(0)

    with torch.no_grad():
        seg = run_segmentation(model, tensor, device='cpu')

    axes[0, i].imshow(img); axes[0, i].axis('off')
    axes[1, i].imshow(mask_gt, cmap='tab10', vmin=0, vmax=3); axes[1, i].axis('off')
    axes[2, i].imshow(seg.mask_full, cmap='tab10', vmin=0, vmax=3); axes[2, i].axis('off')

axes[0, 0].set_ylabel('Image')
axes[1, 0].set_ylabel('GT mask')
axes[2, 0].set_ylabel('Predicted')
plt.suptitle('Segmentation: ground truth vs prediction')
plt.tight_layout()
plt.savefig('../reports/figures/segmentation_qualitative.png', dpi=150)
plt.show()